In [5]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os
import re

In [6]:

# --- utils: simple ingredient cleaner
UNITS = [
    "cup","cups","tbsp","tablespoon","tablespoons","tsp","teaspoon","teaspoons",
    "pound","pounds","lb","lbs","ounce","ounces","oz","grams","g","kg",
    "pinch","clove","cloves","slice","slices","ml","l"
]

units_pattern = r'\b(?:' + '|'.join(re.escape(u) for u in UNITS) + r')\b'
qty_pattern = r'(?:(?:\d+\/\d+)|(?:\d+\.\d+)|(?:\d+))'  # 1, 1.5, 1/2


In [7]:
def clean_ingredient_text(text):
    # text is a list-like string or a joined string of ingredients
    # remove quantities and units, punctuation, lowercase
    # Example: "1 tablespoon soy sauce" -> "soy sauce"
    s = text.lower()
    s = re.sub(r'\(.*?\)', ' ', s)            # remove parentheses
    s = re.sub(qty_pattern, ' ', s)           # remove numeric quantities
    s = re.sub(units_pattern, ' ', s)         # remove units
    s = re.sub(r'[^a-z\s]', ' ', s)           # keep letters only
    s = re.sub(r'\s+', ' ', s).strip()
    return s

In [8]:
df = pd.read_csv("../data/1_Recipe_csv.csv")

df['ingredients'] = df['ingredients'].apply(ast.literal_eval)
# join list into single string (if not already)
df['ingredients_text_raw'] = df['ingredients'].apply(lambda lst: ' '.join(lst))

# clean
df['ingredients_text'] = df['ingredients_text_raw'].apply(clean_ingredient_text)

# optional: drop rows with empty ingredients after cleaning
df = df[df['ingredients_text'].str.strip() != ""]

In [9]:
# TF-IDF with n-grams and sublinear TF
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=15000,     # tune as needed
    ngram_range=(1,2),      # capture "soy sauce"
    sublinear_tf=True,
    min_df=3
)

ingredient_vectors = vectorizer.fit_transform(df['ingredients_text'])

In [10]:
os.makedirs("../models", exist_ok=True)

joblib.dump(vectorizer, "../models/tf_vectorizer.pkl")
joblib.dump(ingredient_vectors, "../models/tf_ingredient_vectors.pkl")
df[['recipe_title', 'ingredients_text']].to_csv("../models/recipes_meta.csv", index=False)

print("✅ TF-IDF model and data saved successfully!")

✅ TF-IDF model and data saved successfully!
